In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

/home/orlandojunior/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-11 19:44:58.027053: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-11 19:44:58.064553: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-11 19:44:58.873764: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly differ

In [2]:
load_dotenv()

True

In [3]:
# Initialize Text Splitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
)

In [8]:
# Get Embeddings Model
embeddings = OpenAIEmbeddings()
# Initialize the LLM instance
llm  = ChatOpenAI(max_tokens=200)

In [5]:
pdf_link = './os-sertoes.pdf'

loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()


In [6]:
len(pages)

658

In [11]:
chunks = text_splitter.split_documents(pages)

# Initialize ChromaDB as Vector Store
db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chromadb',
)


In [16]:
retriever = db.as_retriever(
    search_kwargs={"k":3}
)

prompt_template = """
Você é um especialista no livro os sertões, responda as perguntas com base no contexto apresentado. 
Se você não souber da resposta, diga que você não sabe

context: {context}

question: {query}

answer: """

custom_rag_prompt = PromptTemplate.from_template(prompt_template)

In [17]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | custom_rag_prompt
    | llm
    | StrOutputParser()
)

# Query the RAG Chain
rag_chain.invoke(
  "Quem é o autor do livro os sertões?"
)

'Euclides da Cunha.'